## 3. The DICOM Standard: Structure and Geometry

**DICOM** (Digital Imaging and Communications in Medicine) is the backbone of clinical imaging. Research workflows often convert DICOM into analysis-friendly formats (e.g., **NIfTI**), but the *source of truth* for **geometry** and acquisition metadata is the DICOM header.

**Learning goals**
- Recognize what a DICOM file contains (**metadata** + **pixel data**).
- Identify the 3 tags that control spatial reconstruction (**Pixel Spacing**, **Image Position**, **Image Orientation**).
- Understand how these tags define an **affine** that maps voxel indices $(i,j,k)$ to millimeters $(x,y,z)$.

**Take-home messages**
- Always treat DICOM as a **geometric object**, not just an image file.
- Most downstream failures come from **wrong orientation/spacing**, not from the ML model.

### 3.1 DICOM file anatomy
A DICOM file is a serialized object with two parts:
- **Header (metadata):** patient/study/series identifiers, acquisition settings, and (crucially) geometry.
- **Body (pixel data):** the actual image intensities.

**Key structural idea:** the file is a sequence of **Data Elements**. Each element has:
- **Tag** *(group, element)* — e.g., **(0010,0010)** = Patient Name
- **VR (Value Representation)** — data type (e.g., **DA**, **DS**, **UI**)
- **Length** — bytes in the value
- **Value** — the payload

In practice, you rarely read a *single* DICOM file: you parse a directory, then reconstruct a volume from a **Series** (a stack of slices) within a **Study**.

### 3.2 Geometry-critical DICOM tags
To map voxel indices $(i,j,k)$ to physical coordinates $(x,y,z)$ in millimeters, you need three tags (per slice) plus a definition of slice step:
- **Pixel Spacing (0028,0030):** in-plane spacing $(\Delta y, \Delta x)$ (row, column).
- **Image Position (Patient) (0020,0032):** translation $(T_x, T_y, T_z)$ of the first voxel (top-left) in patient coordinates.
- **Image Orientation (Patient) (0020,0037):** two direction cosine vectors defining row/column axes in 3D:
  - $r_{row}=(r_{xx}, r_{xy}, r_{xz})$ (increasing **column** index)
  - $r_{col}=(r_{yx}, r_{yy}, r_{yz})$ (increasing **row** index)

**Slice direction and spacing:**
- The slice-normal direction is $r_{norm}=r_{row}\times r_{col}$.
- The true slice spacing $\Delta z$ is best derived from differences in **Image Position (Patient)** between adjacent slices (not necessarily **Slice Thickness (0018,0050)**).

### 3.3 Full affine matrix (voxel indices → patient mm)
We use homogeneous coordinates so that **rotation + scaling + translation** are all in one matrix.

Let voxel indices be $V=[i,j,k,1]^T$ and physical coordinates be $P=[x,y,z,1]^T$. Then:
$$P = M \cdot V$$
with the **full** $4\times4$ affine:
$$
M = \begin{bmatrix}
r_{xx}\Delta x & r_{yx}\Delta y & r_{nx}\Delta z & T_x \\
r_{xy}\Delta x & r_{yy}\Delta y & r_{ny}\Delta z & T_y \\
r_{xz}\Delta x & r_{yz}\Delta y & r_{nz}\Delta z & T_z \\
0 & 0 & 0 & 1
\end{bmatrix}
$$
where $r_{norm}=(r_{nx},r_{ny},r_{nz})$.

**How to read this matrix (intuition):**
- The first three columns are the three **basis vectors** of your voxel axes, expressed in mm.
- The last column is the **origin** (translation) in mm.
- Moving by +1 in $i$, $j$, or $k$ moves by one column vector in physical space.

### 3.4 Coordinate conventions: LPS vs RAS (the “silent bug”)
Two common *patient-centered* anatomical conventions:
- **LPS**: +x Left, +y Posterior, +z Superior (common in DICOM/ITK ecosystems).
- **RAS**: +x Right, +y Anterior, +z Superior (common in NIfTI / many research tools).

A simple conversion between the two (for coordinates) is:
$$
\begin{bmatrix}x_{RAS}\\y_{RAS}\\z_{RAS}\\1\end{bmatrix} =
\underbrace{\begin{bmatrix}-1&0&0&0\\0&-1&0&0\\0&0&1&0\\0&0&0&1\end{bmatrix}}_{\text{LPS→RAS flip}}
\begin{bmatrix}x_{LPS}\\y_{LPS}\\z_{LPS}\\1\end{bmatrix}
$$
Apply the same idea to affines: $M_{RAS} = F\,M_{LPS}$, where $F=\mathrm{diag}(-1,-1,1,1)$.

**Take-home messages**
- Geometry lives in the header: **Pixel Spacing + Position + Orientation → affine**.
- Many “mystery” errors are just an **LPS↔RAS sign flip**.

## 4. Analysis formats: from Series to Volumes (NIfTI)
DICOM is optimized for **storage/communication**; ML and numerical pipelines are optimized for **arrays/tensors**. A 3D algorithm (e.g., a 3D U-Net) needs a contiguous volume, not hundreds of individual slice files.

### 4.1 NIfTI (.nii, .nii.gz) in 60 seconds
Why NIfTI is popular for research:
- **Single file volume:** 3D (and 4D) stored as one object (often compressed as **.nii.gz**).
- **Explicit orientation:** header stores affine transforms (**sform/qform**) mapping voxel indices to world coordinates.
- **Tooling ecosystem:** widely supported by Python libraries and neuro/medical imaging software.

### 4.2 Converting DICOM → NIfTI with `dcm2niix`
Conversion is not a file copy: it stacks slices, computes a global affine, and exports relevant metadata.

**Why `dcm2niix`?**
- **Robustness:** handles vendor quirks (Siemens/GE/Philips) well.
- **Sidecar JSON:** can emit BIDS-style metadata that doesn’t fit the NIfTI header.
- **Orientation handling:** standardizes orientation conventions and updates the affine accordingly.

**Example command**
```bash
dcm2niix -z y -f output_filename -o /output/directory /path/to/dicom_directory
```

**Take-home messages**
- Convert once, then train/segment on NIfTI — but keep DICOM around for provenance.
- Always verify the resulting **orientation + spacing** before continuing.

### 4.3 Trust but verify: orientation QA
A flipped axis is the most common (and most damaging) preprocessing error.

**Quick checklist**
- **Visualize:** open the NIfTI in **3D Slicer** or **ITK-SNAP**.
- **Pick an asymmetric landmark:** e.g., in cardiac imaging the **apex direction** and overall laterality; in abdominal scans the **liver** is on the right.
- **Read the orientation labels:** confirm that L/R/A/P/S/I markers match the anatomy you expect.
- **If unsure:** compare to a known-correct case and/or re-check the affine header.

**Take-home message**
- “Looks fine” is not enough — confirm orientation with a real anatomical landmark.